This notebook implements the algorithm discussed in this paper: https://arxiv.org/pdf/2002.08953,
Predicting Many Properties of a Quantum System from Very Few Measurements, by Huang, et al. 

In [ ]:
import classiq
classiq.authenticate()

In [32]:
from classiq import *
import numpy as np
import random
from classiq.execution import ExecutionSession, ExecutionPreferences, ClassiqBackendPreferences
from classiq.execution import ClassiqSimulatorBackendNames
from typing import List
import matplotlib.pyplot as plt
import time

In [60]:
np.random.seed(555) # Set random seed for unitary selection.
# Global Variables: must be initialized globally because they cannot be passed into main.
ids = [] # 
num_qubits = 2
num_snapshots = 100
unitary_ensemble = []

In [61]:
@qfunc(generative=True)
def unitary_application(state: QArray) -> None:
    """
    Applies randomly selected unitaries from a unitary ensemble. 1 per qubit. Updates ids with the identifiers
    of the randomly selected unitaries.

    Args: 
        state (QArray): state prepared by the prep_state function, on which the classical shadow will be calculated.
    Returns:
        None.
    """

    # For the purposes of this example, we consider the application of a random unitary to be a measurement in a random basis.
    # Instead of applying unitaries, we wish to measure in either the X, Y, or Z bases.
    basis_ids = list(np.random.randint(3, size=num_qubits))
    ids.append(basis_ids)

    for i in range(num_qubits):
        #ids of 0, 1, 2 correspond to measurements in the X, Y, and Z basis respectively
        if basis_ids[i] == 0: # H is the change-of-basis matrix from X to Z
            H(state[i])
        if basis_ids[i] == 1: # HS^dagger is the change-of-basis matrix from Y to Z
            SDG(state[i])
            H(state[i])
        # A measurement in the computational basis is a measurement in the Z basis. No unitary application needed.

    # If using a conventional unitary ensemble, uncomment the following code, and disable the code above.
    # num_unitaries = len(unitary_ensemble)
    # unitary_ids = list(np.random.randint(num_unitaries, size=num_qubits))
    # for i in range(num_qubits):
    #     unitary_ensemble[unitary_ids[i]](state[i])

In [62]:
@qfunc
def prep_state(qarr: QArray) -> None:
    """
    Prepares the desired state. The classical shadow will be calculated on this state.

    Args:
        qarr (QArray): Zero initialized array of qubit(s).

    Returns:
        None.
    """
    # For the purposes of this example, this function will prepare the Phi^+ bell state.
    H(qarr[0])
    CX(qarr[0], qarr[1])

In [63]:
@qfunc(generative=True)
def main(qarr: Output[QArray]) -> None:
    """
    Prepares the desired state and applies the randomly selected unitaries once.

    Args:
        None.

    Returns:
        None.
    """
    allocate(num_qubits, qarr)
    prep_state(qarr)
    unitary_application(qarr)

In [64]:
def calculate_shadow(num_snapshots):
    """
    Computes a shadow of size num_snapshots by performing num_snapshots measurements. 

    Args:
        num_snapshots (int): size of shadow, also the number of measurements.

    Returns:
        snapshots (List[List[int]]): List of measurement results (0 or 1) for each circuit of length num_snapshots.
          Inner lists are num_qubits long.
    """
    snapshots=[]
    for i in range(num_snapshots):
        execution_preferences = ExecutionPreferences(
            num_shots=1,
            backend_preferences=ClassiqBackendPreferences(
                backend_name=ClassiqSimulatorBackendNames.SIMULATOR # Set this to best suit your needs.
            ),
            random_seed=np.random.randint(1e6)
        )
        qmod = create_model(main)
        qmod = set_execution_preferences(qmod, execution_preferences)
        qprog = synthesize(qmod)
        job = execute(qprog)
        counts = job.get_sample_result().parsed_counts
        snapshot = list(counts[0].state['qarr'])
        snapshots.append(snapshot) 
    return snapshots

In [65]:
snapshots = calculate_shadow(num_snapshots)
print(f"snapshots: {snapshots}")
print(f"ids: {ids}")

snapshots: [[0, 1], [1, 1], [1, 0], [1, 0], [1, 0], [1, 0], [0, 1], [0, 1], [0, 0], [1, 0], [0, 0], [0, 1], [0, 0], [0, 0], [0, 0], [1, 1], [1, 1], [0, 1], [1, 1], [0, 1], [0, 1], [0, 0], [0, 1], [1, 0], [1, 0], [1, 1], [0, 1], [0, 0], [1, 1], [0, 1], [1, 1], [1, 0], [0, 1], [1, 1], [1, 0], [0, 0], [1, 0], [0, 0], [0, 1], [0, 0], [1, 1], [1, 1], [0, 0], [0, 0], [1, 0], [1, 1], [0, 0], [1, 1], [0, 0], [0, 1], [1, 0], [1, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 1], [0, 1], [0, 0], [1, 0], [0, 0], [0, 0], [0, 1], [0, 1], [0, 0], [1, 0], [0, 1], [1, 1], [0, 0], [0, 1], [0, 1], [0, 1], [1, 1], [1, 0], [1, 1], [0, 0], [0, 1], [1, 1], [1, 0], [0, 0], [0, 0], [0, 1], [0, 1], [1, 1], [1, 0], [0, 0], [1, 0], [0, 0], [1, 1], [0, 1], [0, 0], [1, 0], [1, 1], [1, 1], [0, 0], [1, 1], [1, 1], [1, 1], [0, 0]]
ids: [[2, 1], [0, 1], [2, 1], [0, 1], [0, 2], [1, 2], [0, 2], [0, 1], [2, 2], [0, 1], [0, 0], [1, 0], [0, 1], [0, 0], [1, 0], [0, 0], [0, 0], [1, 0], [0, 0], [0, 1], [2, 1], [0, 0], [2, 1]

In [66]:
def single_snapshot_reconstruction(snapshot, single_ids):
    """
    Reconstructs the state density matrix from a single snapshot. Helper for reconstruction().

    Args:
        snapshot (List[int]): Single element of snapshots.
        single_ids (List[int]): Single element of ids.

    Returns:
        snapshot_state (NumPy array): Single snapshot density matrix reconstruction.

    """

    # Density matrices of computational basis states
    zero_state = np.array([[1, 0], [0, 0]])
    one_state = np.array([[0, 0], [0, 1]])

    # local qubit unitaries
    SDG_matrix = np.array([[1, 0], [0, -1j]], dtype=complex) # Matrix of S^dagger gate.
    hadamard = (1/np.sqrt(2))*np.array([[1, 1], [1, -1]]) # Matrix of Hadamard gate.
    identity = np.array([[1, 0], [0, 1]]) # Matrix of identity gate.

    # Invert the unitaries applied in unitary_application(). 
    unitaries = [hadamard, hadamard@SDG_matrix, identity] # Contents of unitaries depend on unitary_ensemble. 


    snapshot_state = [1]
    for i in range(num_qubits):
        state = zero_state if snapshot[i] == 0 else one_state
        U = unitaries[int(single_ids[i])]

        # Applying Eq. (S44) from Huang, et al.
        local_state = 3 * (U.conjugate().transpose() @ state @ U) - identity
        snapshot_state = np.kron(snapshot_state, local_state)

    return snapshot_state

In [67]:
def reconstruction(in_snapshots, in_ids):
    """
    Performs a full reconstruction of the quantum state density matrix, where the quantum state is that
    prepared in prep_state().

    Args:
        in_snapshots (List[List[int]]): List of snapshots. Will most likely be the global `snapshots`.
        in_ids (List[List[int]]): List of unitary identifiers. Will most likely be the global `ids`.

    Returns:
        shadow_state (NumPy array): Full state density matrix reconstruction.
    """ 
    # Average over snapshots.
    shadow_state = np.zeros((2 ** num_qubits, 2 ** num_qubits), dtype=complex)
    for i in range(num_snapshots):
        result = single_snapshot_reconstruction(in_snapshots[i], in_ids[i])
        shadow_state += result

    return shadow_state/num_snapshots

In [68]:
shadow_state = reconstruction(snapshots, ids)
print(f"Full state density matrix: \n {np.round(shadow_state, decimals=6)}") # Rounding for convenience.

Full state density matrix: 
 [[ 0.4075+0.j      0.105 +0.1575j  0.075 -0.0225j  0.4725+0.j    ]
 [ 0.105 -0.1575j  0.1075+0.j      0.1575+0.09j   -0.015 -0.1125j]
 [ 0.075 +0.0225j  0.1575-0.09j    0.0775+0.j     -0.03  -0.1575j]
 [ 0.4725-0.j     -0.015 +0.1125j -0.03  +0.1575j  0.4075+0.j    ]]


In [69]:
bell_state_density_matrix = np.array([[0.5, 0, 0, 0.5], [0, 0, 0, 0], [0, 0, 0, 0], [0.5, 0, 0, 0.5]])

ref_state_density_matrix = bell_state_density_matrix # Set this to whatever your reference state density matrix is. 

def norm(ref_state, estimated_state):
    """
    Computes the Frobenius norm of a state, i.e. the distance between two operators.

    Args:
        state (NumPy arra.y): difference of estimated and reconstructed states.
        ref_state_density_matrix (NumPy array): Density matrix of reference state.
        estimated_state (NumPy array): Density matrix of estimated state.
    
    Returns:
        norm (NumPy complex number): Distance between the states.
    """

    state = ref_state - estimated_state
    return np.sqrt(np.trace(state.conjugate().transpose() @ state))

In [70]:
print(f"Comparing to: \n {bell_state_density_matrix}")
print(f"Distance: {norm(ref_state_density_matrix, shadow_state)}")

Comparing to: 
 [[0.5 0.  0.  0.5]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.  0.5]]
Distance: (0.5129327441292862+0j)


In [ ]:
def estimate_observable(in_snapshots, in_ids, div): #observable parameter excluded for testing
    """
    Estimates the expectation value of an observable with respect the state prepared in prep_state().
    As of now, the observable cannot be passed in. The lists representing it must be manually inputted below.

    Args:
        in_snapshots (List[List[int]]): List of snapshots. Will most likely be the global `snapshots`.
        in_ids (List[List[int]]): List of unitary identifiers. Will most likely be the global `ids`.
        div (int): number of divisions for median-of-means.

    Returns:
        expval (float): Esimated expectation value.
    """
    means = []
    in_snapshots = np.array(in_snapshots)
    in_snapshots = 1 - 2*in_snapshots #change snapshots from containing tuples with {0, 1} to {1, -1} (respective state eigenvalues)
    in_ids = np.array(in_ids)
    # target obs = [] #paulis in observable
    # target_locs = [] #qubits the above paulis act on

    #pseudocode to populate the above arrays:
    #if observable acts on a single qubit at a time, determine which, and populate the lists
    #if not, 
    #   for each operand in the observable, add it to target_obs. the wire that operand acts on is at the same index
    #   in target_locs

    #assume that observables can be decoded in that way.

    #As an example, we will decode np.kron(Z, Z), for which the expected result is 1.0.

    target_obs = [2, 2] # Z basis has identifier 2.
    target_locs = [0, 1] # Z gate acts on both qubits.

    # To estimate the expectation value of an observable, we must compute the trace of the final density matrix.
    # Therefore, we are simply computing the trace of Eq. S44. 
    # We can take a shortcut: Due to the orthogonality of the Pauli operators,
    # it evaluates to ±3 if the Pauli bases of the observable match the corresponding measurement basis, and evaluates to 0 otherwise.


    # Divide in_snapshots and in_ids into div chunks.
    for i in range(0, num_snapshots, num_snapshots//div):
        in_snapshots_div, in_ids_div = (
            in_snapshots[i: i + num_snapshots // div],
            in_ids[i: i + num_snapshots // div],
        )

        # Select all sets of ids that match type and position of Pauli operators
        indices = np.all(in_ids_div[:, target_locs] == target_obs, axis=1)

        if sum(indices) > 0:
            # Take the product and sum
            product = np.prod(in_snapshots_div[indices][:, target_locs], axis=1)
            means.append(np.sum(product) / sum(indices))
        else:
            means.append(0)


    return np.median(means)

In [ ]:
print(f"Estimated expectation value: {estimate_observable(snapshots, ids, div=8)}") 
#Due to the low number of snapshots, this may vary from run to run.

Estimated expectation value: 1.0


In [73]:
def error_bound(error, observables, failure_rate=0.01):
    """
    Calculate the shadow bound for the Pauli measurement scheme.

    Implements Eq. (S13) from Huang, et al.

    Args:
        error (float): The error on the estimator.
        observables (list) : List of matrices corresponding to the observables we intend to measure.
        failure_rate (float): Rate of failure for the bound to hold.

    Returns:
        An integer that gives the number of samples required to satisfy the shadow bound and
        the chunk size required attaining the specified failure rate. 
    """
    M = len(observables)
    K = 2 * np.log(2 * M / failure_rate)
    shadow_norm = (
        lambda op: np.linalg.norm(
            op - np.trace(op) / 2 ** int(np.log2(op.shape[0])), ord=np.inf
        )
        ** 2
    )
    N = 34 * max(shadow_norm(o) for o in observables) / error ** 2
    return int(np.ceil(N * K)), int(K)

In [ ]:
z_gate = np.array([[1, 0], [0, -1]])
o = np.kron(z_gate, z_gate)
observables = []
observables.append(o)
required_snapshots = error_bound(0.5, observables)
print(f"Number of samples required, number of divisions: {required_snapshots}") 

Number of samples required, number of divisions: (1442, 10)
